structured output

using pydentic

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, client=<groq.resources.chat.completions.Completions object at 0x000001EC06B4F0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EC06B4FB60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the Movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The Director of the movie")
    rating:float=Field(description="The movies ratings out of 10")    

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, client=<groq.resources.chat.completions.Completions object at 0x000001EC06B4F0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EC06B4FB60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the Movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The Director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'M

In [14]:
response=model.invoke("provide details about the movie batman")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request:** The user asked for "details about the movie batman". This is quite broad since there are multiple Batman movies. I need to clarify which one they mean, but also provide a comprehensive overview that covers the major ones, or ask for clarification while giving a structured response.\n\n2.  **Identify Key Ambiguity:** "Batman" has been adapted into numerous films across different eras and franchises:\n   - Tim Burton\'s Batman (1989)\n   - Batman Returns (1992)\n   - Batman Forever (1995)\n   - Batman & Robin (1997)\n   - Christopher Nolan\'s The Dark Knight Trilogy (2005-2012)\n   - DC Extended Universe (DCEU) films: Batman v Superman (2016), Suicide Squad (cameo), Justice League (2017)\n   - Matt Reeves\' The Batman (2022)\n   - Animated films (many)\n   - Upcoming: The Batman Part II (2026)\n\n3.  **Determine Response Strategy:** \n   - Acknowledge the ambiguity\n   - Provide a structured ove

In [13]:
response1=model_with_structure.invoke("provide details about the movie batman begins")
response1

Movie(title='Batman Begins', year=2005, director='Christopher Nolan', rating=8.2)

Message output parsed structure

In [15]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the Movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The Director of the movie")
    rating:float=Field(...,description="The movies ratings out of 10")  

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response1=model_with_structure.invoke("provide details about the movie batman begins")
response1  

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "provide details about the movie batman begins"\n   - Key entity: "Batman Begins" (movie)\n   - Expected details: title, year, director, rating (based on the available function schema)\n\n2.  **Identify Available Function:**\n   - Function: `Movie`\n   - Parameters: `title` (string, required), `year` (integer, required), `director` (string, required), `rating` (number, required)\n\n3.  **Determine Required Information for Function Call:**\n   - I need to provide all four required parameters: title, year, director, rating.\n   - The user only provided the title ("batman begins").\n   - I need to fill in the missing information based on my knowledge:\n     - Title: "Batman Begins"\n     - Year: 2005\n     - Director: Christopher Nolan\n     - Rating: I need to provide a rating out of 10. Common ratings for Batman Begins: IMDb ~8.2, Rotten Toma

nested structure

In [19]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None,description="Budget is millions USD")

model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("provide me the detail of movie Batman Begins")
response
    

Movie(title='Batman Begins', year=2005, cast=[Actor(name='Christian Bale', role='Bruce Wayne / Batman'), Actor(name='Michael Caine', role='Alfred Pennyworth'), Actor(name='Liam Neeson', role="Henri Ducard / Ra's al Ghul"), Actor(name='Katie Holmes', role='Rachel Dawes'), Actor(name='Gary Oldman', role='Commissioner James Gordon'), Actor(name='Morgan Freeman', role='Lucius Fox'), Actor(name='Cillian Murphy', role='Dr. Jonathan Crane / Scarecrow')], genres=['Action', 'Crime', 'Drama'], budget=150.0)

TypeDict

In [ ]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str,...,"the title of the movie"]
    year:Annotated[str,...,"the year of the movie"]
    director:Annotated[str,...,"the director of the movie"]
    ratings:Annotated[str,...,"the ratings of the movie"]

model_with_structure=model.with_structured_output(MovieDict)
response=model_with_structure.invoke("provide me the detail of movie Batman Begins")
response
    

{'director': 'Christopher Nolan',
 'ratings': '8.2/10',
 'title': 'Batman Begins',
 'year': '2005'}

In [23]:

class Actor(TypedDict):
    name:str
    role:str

class Movie(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None,description="Budget is millions USD")

model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("provide me the detail of movie Batman Begins")
response
    

{'budget': 150000000,
 'cast': [{'name': 'Christian Bale', 'role': 'Bruce Wayne / Batman'},
  {'name': 'Michael Caine', 'role': 'Alfred Pennyworth'},
  {'name': 'Liam Neeson', 'role': "Henri Ducard / Ra's al Ghul"},
  {'name': 'Katie Holmes', 'role': 'Rachel Dawes'},
  {'name': 'Gary Oldman', 'role': 'Commissioner James Gordon'},
  {'name': 'Morgan Freeman', 'role': 'Lucius Fox'},
  {'name': 'Cillian Murphy', 'role': 'Dr. Jonathan Crane / Scarecrow'}],
 'genres': ['Action', 'Crime', 'Drama'],
 'title': 'Batman Begins',
 'year': 2005}

In [26]:
model.profile


Data Classes

In [ ]:
# pydantic
import os
from langchain.agents import create_agent
from pydantic import BaseModel,Field

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

class ContactInfo(BaseModel):
    """contact information of a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:int=Field(description="The phone number of the person")

agent=create_agent(
    model="google_genai:gemini-3.6-flash",
    response_format=ContactInfo     #auto selected provider startergy
    )

result=agent.invoke({"messages":[{"role":"user","content":"Extract contact info from :John Doe,johndoe123@gmail.com,8787878787"}]})

result



{'messages': [HumanMessage(content='Extract contact info from :John Doe,johndoe123@gmail.com,8787878787', additional_kwargs={}, response_metadata={}, id='120c9ee8-2df0-4539-8a18-452b23514847'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"johndoe123@gmail.com","phone":8787878787}', 'extras': {'signature': 'EsQICsEIARFNMg+qiXvEb8nxb8ff64wLVONkkVWRNYYvLOW9WhuO6h+5pPsB2gWtBGCQV9PVv1K02YEL2Srqkt2v+5iwm5+Xspku2mqXzq5vidM8u+aFOUe4R+2syu0lzBx3b3F3L/tPCnbuxNVdqt94hSQOx34hsTsOFKcpGARjnmejcHNTDvYYYMdtgX7B0O68xAsPwlRYvAWTSGHe67WTAqDRnLWhoFaFrE5h2Nej0oOfSNxJbcbze4H1AlCkVIYiaWlZ1NbssolUf2c/aNb32FaH8QFY0+VAfzUf/VSYZ3znI9Ej52NB1vyluotPqWRJ4neBS7axmv0XH4qYhPDSUFxhZo6LASjgFkJrj5qVv+PSmbHdeLCGECI3uhtaYn4ZDNwPXWyIdX2i0IxviFOli9MUpvKNClW4ytn2xiglsW3qi0f6i2U/Txn06CM5Q3FPBh5yVNxXdn4MEFhiUJsJg29mVGDTS2neD73Hq8mrEWg8mHkY6dIWOUlg2lRwvq/U18LnrSfVPkhB4Z1CtsisveuD546nInzrUOGG8d04XBGVb+zEyTYW+WyUHGSFxgsPJi/GuWthN+OXGohxg5Au9kzpc9Jw62qxTsYu/wUkd75tMNFCl7ryaMA22aIFSIX1TfCrcXfHs5m4PojNak

In [31]:
print(result["structured_response"])

name='John Doe' email='johndoe123@gmail.com' phone=8787878787


In [32]:
#typedDict
import os
from langchain.agents import create_agent
from typing_extensions import TypedDict,Annotated

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

class ContactInfo(TypedDict):
    """contact information of a person"""
    name:str #name of the person
    email:str # email address of the person
    phone:int #phone number of the person

agent=create_agent(
    model="google_genai:gemini-3.6-flash",
    response_format=ContactInfo     #auto selected provider startergy
    )

result=agent.invoke({"messages":[{"role":"user","content":"Extract contact info from :John Doe,johndoe123@gmail.com,8787878787"}]})

result["structured_response"]



{'name': 'John Doe', 'email': 'johndoe123@gmail.com', 'phone': 8787878787}

In [2]:
#data class
import os
from langchain.agents import create_agent
from dataclasses import dataclass

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

@dataclass
class ContactInfo:
    """contact information of a person"""
    name:str #The name of the person"
    email:str #"The email address of the person"
    phone:int #"The phone number of the person"

agent=create_agent(
    model="google_genai:gemini-3.6-flash",
    response_format=ContactInfo     #auto selected provider startergy
    )

result=agent.invoke({"messages":[{"role":"user","content":"Extract contact info from :John Doe,johndoe123@gmail.com,8787878787"}]})

result["structured_response"]



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ContactInfo(name='John Doe', email='johndoe123@gmail.com', phone=8787878787)